# Contextual Grounding

Guardrail policies so far have focused on keeping bad content **out** — blocking harmful 
requests, filtering PII, rejecting off-topic queries. Contextual grounding flips the 
perspective: it checks whether the model's **output** is actually supported by the source 
material you provide.

Two checks run on every response:
- **Grounding score** — Is the model's claim backed by the reference text? Catches hallucinations.
- **Relevance score** — Does the response actually answer the question asked? Catches drift.

Each score has a configurable threshold (0.0–0.99). Responses that fall below the threshold 
get flagged or blocked.

**Insurance context:** When a customer asks "does my policy cover water damage?", the answer 
must come from *their actual policy document*, not the model's general training knowledge. 
Contextual grounding enforces that.

In [39]:
# ============================================================
# Setup and Configuration
# ============================================================

import boto3
import json
import random
import string
import time

bedrock = boto3.client('bedrock', region_name='us-east-1')
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Retrieve our existing guardrail from Day 5
GUARDRAIL_ID = 'your-guardrail-id'  # from your AWS console or previous notebooks

existing = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')

print(f"Guardrail: {existing['name']}")
#print(f"Guardrail ID: {GUARDRAIL_ID}")
print(f"Status: {existing['status']}")
print(f"Model: {MODEL_ID}")

Guardrail: insurance-assistant-guardrail
Status: READY
Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0


In [40]:
# ============================================================
# Cell 3: Add Contextual Grounding Policy
# ============================================================
# Contextual grounding checks model OUTPUT
# Two thresholds:
#   - groundingThreshold: is the response supported by the source?
#   - relevanceThreshold: does the response answer the question?
# Scale is 0.0 to 0.99. Higher = stricter.


# Remap GET keys to UPDATE keys
content_policy = existing.get('contentPolicy', {})
topic_policy = existing.get('topicPolicy', {})
sensitive_policy = existing.get('sensitiveInformationPolicy', {})
word_policy = existing.get('wordPolicy', {})

response = bedrock.update_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    name=existing['name'],
    blockedInputMessaging=existing['blockedInputMessaging'],
    blockedOutputsMessaging=existing['blockedOutputsMessaging'],
    contentPolicyConfig={
        'filtersConfig': content_policy.get('filters', [])
    },
    topicPolicyConfig={
        'topicsConfig': topic_policy.get('topics', [])
    },
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': sensitive_policy.get('piiEntities', []),
        'regexesConfig': sensitive_policy.get('regexes', [])
    },
    wordPolicyConfig={
        'wordsConfig': word_policy.get('words', []),
        'managedWordListsConfig': word_policy.get('managedWordLists', [])
    },
    # NEW: Add contextual grounding
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.7
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.7
            }
        ]
    }
)

#print(f"Updated guardrail: {response['guardrailId']}")
print(f"Version: {response['version']}")

# ---- Sanity check: verify all policies survived ----
updated = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')

filters = updated.get('contentPolicy', {}).get('filters', [])
print("\nCONTENT FILTERS:")
for f in filters:
    print(f"  {f['type']}: input={f.get('inputStrength','—')} output={f.get('outputStrength','—')}")

topics = updated.get('topicPolicy', {}).get('topics', [])
print(f"\nDENIED TOPICS ({len(topics)}):")
for t in topics:
    print(f"  {t['name']}")

pii = updated.get('sensitiveInformationPolicy', {})
entities = pii.get('piiEntities', [])
regexes = pii.get('regexes', [])
print(f"\nPII DETECTORS: {len(entities)} built-in, {len(regexes)} custom regex")

words = updated.get('wordPolicy', {})
custom = words.get('words', [])
managed = words.get('managedWordLists', [])
print(f"\nWORD FILTERS: {len(custom)} custom words, {len(managed)} managed lists")

Version: DRAFT

CONTENT FILTERS:
  VIOLENCE: input=LOW output=MEDIUM
  PROMPT_ATTACK: input=HIGH output=NONE
  MISCONDUCT: input=MEDIUM output=HIGH
  HATE: input=HIGH output=HIGH
  SEXUAL: input=HIGH output=HIGH
  INSULTS: input=LOW output=HIGH

DENIED TOPICS (6):
  Investment Advice
  Medical Diagnosis
  Legal Advice
  Coverage Guarantees
  Competitor Comparisons
  Claim Value Adjustments

PII DETECTORS: 10 built-in, 3 custom regex

WORD FILTERS: 12 custom words, 1 managed lists


In [41]:
grounding = updated.get('contextualGroundingPolicy', {})
print(grounding)

{'filters': [{'type': 'GROUNDING', 'threshold': 0.7}, {'type': 'RELEVANCE', 'threshold': 0.7}]}


In [28]:
# ============================================================
# Sample Reference Document
# ============================================================
# In production, this would come from a Knowledge Base .
# For now, we provide it manually to see how grounding works.

POLICY_DOCUMENT = """
INSURANCE POLICY: AUTO COVERAGE SUMMARY
Policy Type: Comprehensive Auto Insurance
Policy Number: POL-2024-ON-445566

COVERAGE DETAILS:
1. Collision Coverage: Covers damage to your vehicle from collisions with 
   other vehicles or objects. Deductible: $500. Maximum payout: $50,000.

2. Comprehensive Coverage: Covers non-collision damage including theft, 
   vandalism, fire, natural disasters, and falling objects. 
   Deductible: $250. Maximum payout: $50,000.

3. Liability Coverage: Bodily injury liability up to $1,000,000 per 
   occurrence. Property damage liability up to $500,000 per occurrence.

4. Uninsured Motorist: Coverage up to $500,000 for accidents involving 
   uninsured or underinsured drivers.

5. Rental Car Coverage: Up to $40 per day for a maximum of 30 days 
   while your vehicle is being repaired for a covered claim.

EXCLUSIONS:
- Intentional damage to your own vehicle
- Damage from racing or competitive driving events
- Wear and tear or mechanical breakdown
- Damage occurring while vehicle is used for commercial delivery services

CLAIMS PROCESS:
- Report the incident within 48 hours
- Provide photos of damage and a police report if applicable
- An adjuster will assess the damage within 5 business days
- Approved claims are paid within 10 business days of assessment
"""

print(f"Reference document loaded: {len(POLICY_DOCUMENT)} characters")
print("This simulates a policy document retrieved from a Knowledge Base")

Reference document loaded: 1308 characters
This simulates a policy document retrieved from a Knowledge Base


In [29]:
# ============================================================
# Cell 6b: Test with ApplyGuardrail API
# ============================================================
# Let's try the ApplyGuardrail API which has explicit support
# for grounding_source, query, and guard_content qualifiers.
# This separates the guardrail check from the model invocation.

# Step 1: Invoke the model normally (without guardrail)
tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
query = "What is my collision coverage deductible?"

body = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'system': (
        "You are an insurance claims assistant. Answer the customer's question "
        "using ONLY the information in the provided policy document. "
        "Do not use any outside knowledge.\n\n"
        f"POLICY DOCUMENT:\n{POLICY_DOCUMENT}"
    ),
    'messages': [{'role': 'user', 'content': query}]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body)
)
model_response = json.loads(response['body'].read())['content'][0]['text']
print("MODEL RESPONSE:", model_response[:200])

# Step 2: Apply guardrail with explicit grounding source
guardrail_result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion='DRAFT',
    source='OUTPUT',
    content=[
        {
            'text': {
                'text': POLICY_DOCUMENT,
                'qualifiers': ['grounding_source']
            }
        },
        {
            'text': {
                'text': query,
                'qualifiers': ['query']
            }
        },
        {
            'text': {
                'text': model_response,
                'qualifiers': ['guard_content']
            }
        }
    ]
)

print(f"\nGUARDRAIL ACTION: {guardrail_result['action']}")
print(f"\nASSESSMENTS:")
for assessment in guardrail_result.get('assessments', []):
    if 'contextualGroundingPolicy' in assessment:
        for f in assessment['contextualGroundingPolicy']['filters']:
            print(f"  {f['type']}: score={f['score']}, threshold={f['threshold']}, action={f['action']}")

print(f"\nFull output: {json.dumps(guardrail_result, indent=2, default=str)[:2000]}")

MODEL RESPONSE: According to your policy document, your collision coverage deductible is **$500**.

GUARDRAIL ACTION: NONE

ASSESSMENTS:
  GROUNDING: score=0.98, threshold=0.7, action=NONE
  RELEVANCE: score=1.0, threshold=0.7, action=NONE

Full output: {
  "ResponseMetadata": {
    "RequestId": "082e895e-0194-4632-bdd6-7a16a27d17b0",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Wed, 08 Apr 2026 14:59:56 GMT",
      "content-type": "application/json",
      "content-length": "1436",
      "connection": "keep-alive",
      "x-amzn-requestid": "082e895e-0194-4632-bdd6-7a16a27d17b0"
    },
    "RetryAttempts": 0
  },
  "usage": {
    "topicPolicyUnits": 1,
    "contentPolicyUnits": 1,
    "wordPolicyUnits": 1,
    "sensitiveInformationPolicyUnits": 1,
    "sensitiveInformationPolicyFreeUnits": 1,
    "contextualGroundingPolicyUnits": 2
  },
  "action": "NONE",
  "outputs": [],
  "assessments": [
    {
      "contextualGroundingPolicy": {
        "filters": [
          {


In [31]:
# ============================================================
# Cell 7: Hallucination Test — Ungrounded Response
# ============================================================
# Instead of letting the model respond, we'll feed a fabricated
# response to the guardrail to see it catch ungrounded claims.

query = "What is my collision coverage deductible?"

# This response contains claims NOT in the policy document
fabricated_response = (
    "Your collision coverage deductible is $500. Additionally, you have "
    "a loyalty discount of 15% applied to your premium, and your policy "
    "includes free roadside assistance with unlimited towing up to 200km."
)

guardrail_result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion='DRAFT',
    source='OUTPUT',
    content=[
        {
            'text': {
                'text': POLICY_DOCUMENT,
                'qualifiers': ['grounding_source']
            }
        },
        {
            'text': {
                'text': query,
                'qualifiers': ['query']
            }
        },
        {
            'text': {
                'text': fabricated_response,
                'qualifiers': ['guard_content']
            }
        }
    ]
)

print("FABRICATED RESPONSE:", fabricated_response)
print(f"\nGUARDRAIL ACTION: {guardrail_result['action']}")
print(f"\nASSESSMENTS:")
for assessment in guardrail_result.get('assessments', []):
    if 'contextualGroundingPolicy' in assessment:
        for f in assessment['contextualGroundingPolicy']['filters']:
            print(f"  {f['type']}: score={f['score']}, threshold={f['threshold']}, action={f['action']}")

FABRICATED RESPONSE: Your collision coverage deductible is $500. Additionally, you have a loyalty discount of 15% applied to your premium, and your policy includes free roadside assistance with unlimited towing up to 200km.

GUARDRAIL ACTION: GUARDRAIL_INTERVENED

ASSESSMENTS:
  GROUNDING: score=0.04, threshold=0.7, action=BLOCKED
  RELEVANCE: score=1.0, threshold=0.7, action=NONE


In [32]:
# ============================================================
# Cell 8: Irrelevant Response Test
# ============================================================
# The response contains only facts from the document, but
# answers the WRONG question. Grounding should pass,
# relevance should fail.

query = "What is my collision coverage deductible?"

irrelevant_response = (
    "You must report any incident within 48 hours. Please provide "
    "photos of the damage and a police report if applicable. An adjuster "
    "will assess the damage within 5 business days."
)

guardrail_result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion='DRAFT',
    source='OUTPUT',
    content=[
        {
            'text': {
                'text': POLICY_DOCUMENT,
                'qualifiers': ['grounding_source']
            }
        },
        {
            'text': {
                'text': query,
                'qualifiers': ['query']
            }
        },
        {
            'text': {
                'text': irrelevant_response,
                'qualifiers': ['guard_content']
            }
        }
    ]
)

print("QUERY:", query)
print("RESPONSE:", irrelevant_response)
print(f"\nGUARDRAIL ACTION: {guardrail_result['action']}")
print(f"\nASSESSMENTS:")
for assessment in guardrail_result.get('assessments', []):
    if 'contextualGroundingPolicy' in assessment:
        for f in assessment['contextualGroundingPolicy']['filters']:
            print(f"  {f['type']}: score={f['score']}, threshold={f['threshold']}, action={f['action']}")

QUERY: What is my collision coverage deductible?
RESPONSE: You must report any incident within 48 hours. Please provide photos of the damage and a police report if applicable. An adjuster will assess the damage within 5 business days.

GUARDRAIL ACTION: GUARDRAIL_INTERVENED

ASSESSMENTS:
  GROUNDING: score=0.98, threshold=0.7, action=NONE
  RELEVANCE: score=5e-324, threshold=0.7, action=BLOCKED


In [33]:
# ============================================================
# Cell 9: Threshold Tuning — Borderline Response
# ============================================================
# A response that's mostly grounded but adds a small detail
# not in the document. We'll test at different thresholds
# to see where it tips from PASS to BLOCKED.

query = "How long do I have to report an incident?"

# Mostly correct, but "online portal" is not in the document
borderline_response = (
    "You must report any incident within 48 hours. You can do this "
    "through our online portal or by calling our claims department."
)

thresholds_to_test = [0.5, 0.6, 0.7, 0.8, 0.9, 0.99]

print(f"QUERY: {query}")
print(f"RESPONSE: {borderline_response}")
print(f"\n{'Threshold':<12} {'Grounding':<12} {'G-Action':<12} {'Relevance':<12} {'R-Action':<12}")
print("-" * 60)

for threshold in thresholds_to_test:
    # Update grounding threshold
    existing = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')
    content_policy = existing.get('contentPolicy', {})
    topic_policy = existing.get('topicPolicy', {})
    sensitive_policy = existing.get('sensitiveInformationPolicy', {})
    word_policy = existing.get('wordPolicy', {})

    bedrock.update_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        name=existing['name'],
        blockedInputMessaging=existing['blockedInputMessaging'],
        blockedOutputsMessaging=existing['blockedOutputsMessaging'],
        contentPolicyConfig={'filtersConfig': content_policy.get('filters', [])},
        topicPolicyConfig={'topicsConfig': topic_policy.get('topics', [])},
        sensitiveInformationPolicyConfig={
            'piiEntitiesConfig': sensitive_policy.get('piiEntities', []),
            'regexesConfig': sensitive_policy.get('regexes', [])
        },
        wordPolicyConfig={
            'wordsConfig': word_policy.get('words', []),
            'managedWordListsConfig': word_policy.get('managedWordLists', [])
        },
        contextualGroundingPolicyConfig={
            'filtersConfig': [
                {'type': 'GROUNDING', 'threshold': threshold},
                {'type': 'RELEVANCE', 'threshold': threshold}
            ]
        }
    )
    time.sleep(1)  # Let the update propagate

    result = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion='DRAFT',
        source='OUTPUT',
        content=[
            {'text': {'text': POLICY_DOCUMENT, 'qualifiers': ['grounding_source']}},
            {'text': {'text': query, 'qualifiers': ['query']}},
            {'text': {'text': borderline_response, 'qualifiers': ['guard_content']}}
        ]
    )

    for assessment in result.get('assessments', []):
        if 'contextualGroundingPolicy' in assessment:
            filters = assessment['contextualGroundingPolicy']['filters']
            g = next(f for f in filters if f['type'] == 'GROUNDING')
            r = next(f for f in filters if f['type'] == 'RELEVANCE')
            print(f"{threshold:<12} {g['score']:<12} {g['action']:<12} {r['score']:<12} {r['action']:<12}")

# Reset to 0.7
existing = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion='DRAFT')
content_policy = existing.get('contentPolicy', {})
topic_policy = existing.get('topicPolicy', {})
sensitive_policy = existing.get('sensitiveInformationPolicy', {})
word_policy = existing.get('wordPolicy', {})

bedrock.update_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    name=existing['name'],
    blockedInputMessaging=existing['blockedInputMessaging'],
    blockedOutputsMessaging=existing['blockedOutputsMessaging'],
    contentPolicyConfig={'filtersConfig': content_policy.get('filters', [])},
    topicPolicyConfig={'topicsConfig': topic_policy.get('topics', [])},
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': sensitive_policy.get('piiEntities', []),
        'regexesConfig': sensitive_policy.get('regexes', [])
    },
    wordPolicyConfig={
        'wordsConfig': word_policy.get('words', []),
        'managedWordListsConfig': word_policy.get('managedWordLists', [])
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {'type': 'GROUNDING', 'threshold': 0.7},
            {'type': 'RELEVANCE', 'threshold': 0.7}
        ]
    }
)
print("\n✅ Threshold reset to 0.7")

QUERY: How long do I have to report an incident?
RESPONSE: You must report any incident within 48 hours. You can do this through our online portal or by calling our claims department.

Threshold    Grounding    G-Action     Relevance    R-Action    
------------------------------------------------------------
0.5          0.11         BLOCKED      1.0          NONE        
0.6          0.11         BLOCKED      1.0          NONE        
0.7          0.11         BLOCKED      1.0          NONE        
0.8          0.11         BLOCKED      1.0          NONE        
0.9          0.11         BLOCKED      1.0          NONE        
0.99         0.11         BLOCKED      1.0          NONE        

✅ Threshold reset to 0.7


In [34]:
# ============================================================
# Cell 10: Subtler Borderline — Mild Embellishment
# ============================================================
# A response that's almost entirely from the document but adds
# just a tiny bit of helpful framing not in the source.

query = "What does my comprehensive coverage include?"

# Everything here is from the document EXCEPT "weather-related damage"
# which is an inference from "natural disasters" but not the exact term
subtle_response = (
    "Your comprehensive coverage covers non-collision damage including "
    "theft, vandalism, fire, weather-related damage, and falling objects. "
    "Your deductible is $250 with a maximum payout of $50,000."
)

result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion='DRAFT',
    source='OUTPUT',
    content=[
        {'text': {'text': POLICY_DOCUMENT, 'qualifiers': ['grounding_source']}},
        {'text': {'text': query, 'qualifiers': ['query']}},
        {'text': {'text': subtle_response, 'qualifiers': ['guard_content']}}
    ]
)

print(f"QUERY: {query}")
print(f"RESPONSE: {subtle_response}")
print(f"\nGUARDRAIL ACTION: {result['action']}")
for assessment in result.get('assessments', []):
    if 'contextualGroundingPolicy' in assessment:
        for f in assessment['contextualGroundingPolicy']['filters']:
            print(f"  {f['type']}: score={f['score']}, threshold={f['threshold']}, action={f['action']}")

QUERY: What does my comprehensive coverage include?
RESPONSE: Your comprehensive coverage covers non-collision damage including theft, vandalism, fire, weather-related damage, and falling objects. Your deductible is $250 with a maximum payout of $50,000.

GUARDRAIL ACTION: NONE
  GROUNDING: score=1.0, threshold=0.7, action=NONE
  RELEVANCE: score=1.0, threshold=0.7, action=NONE


In [36]:
# ============================================================
# Cell 11: Mixed Response — Grounded + One Subtle Hallucination
# ============================================================
# Mostly accurate, but slips in one plausible-sounding detail
# that isn't in the policy document.

query = "What happens after I file a claim?"

# The 48 hours, photos, adjuster in 5 days, paid in 10 days — all grounded.
# "dedicated claims representative" — NOT in the document.
mixed_response = (
    "After you report the incident within 48 hours and provide photos "
    "and a police report if applicable, you'll be assigned a dedicated "
    "claims representative. An adjuster will assess the damage within "
    "5 business days, and approved claims are paid within 10 business "
    "days of assessment."
)

result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion='DRAFT',
    source='OUTPUT',
    content=[
        {'text': {'text': POLICY_DOCUMENT, 'qualifiers': ['grounding_source']}},
        {'text': {'text': query, 'qualifiers': ['query']}},
        {'text': {'text': mixed_response, 'qualifiers': ['guard_content']}}
    ]
)

print(f"QUERY: {query}")
print(f"RESPONSE: {mixed_response}")
print(f"\nGUARDRAIL ACTION: {result['action']}")
for assessment in result.get('assessments', []):
    if 'contextualGroundingPolicy' in assessment:
        for f in assessment['contextualGroundingPolicy']['filters']:
            print(f"  {f['type']}: score={f['score']}, threshold={f['threshold']}, action={f['action']}")

QUERY: What happens after I file a claim?
RESPONSE: After you report the incident within 48 hours and provide photos and a police report if applicable, you'll be assigned a dedicated claims representative. An adjuster will assess the damage within 5 business days, and approved claims are paid within 10 business days of assessment.

GUARDRAIL ACTION: GUARDRAIL_INTERVENED
  GROUNDING: score=0.05, threshold=0.7, action=BLOCKED
  RELEVANCE: score=1.0, threshold=0.7, action=NONE


In [37]:
# ============================================================
# Cell 12: Production Grounded Invocation Function
# ============================================================
# Two-step flow:
#   1. Invoke the model with the reference document as context
#   2. Check the response against the reference using apply_guardrail

def invoke_with_grounding(query, reference_doc, system_prompt=None):
    """
    Invoke model with a reference document, then verify the response
    is grounded in that document and relevant to the query.
    Returns status: 'success', 'ungrounded', 'irrelevant', 'blocked', or 'error'
    """
    # Step 1: Invoke the model
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    tagged_content = (
        f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        f'{query}'
        f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
    )
    
    default_system = (
        "You are an insurance claims assistant. Answer the customer's question "
        "using ONLY the information in the provided document. "
        "Do not use any outside knowledge.\n\n"
        f"REFERENCE DOCUMENT:\n{reference_doc}"
    )
    
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 1024,
        'amazon-bedrock-guardrailConfig': {'tagSuffix': tag_suffix},
        'system': system_prompt or default_system,
        'messages': [{'role': 'user', 'content': tagged_content}]
    }

    try:
        # Invoke with guardrails (content filters, denied topics, PII, word filters)
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            body=json.dumps(body),
            trace='ENABLED'
        )
        result = json.loads(response['body'].read())
        model_response = result['content'][0]['text']
        guardrail_action = result.get('amazon-bedrock-guardrailAction', 'NONE')

        # If the first-pass guardrails blocked it, return immediately
        if guardrail_action == 'INTERVENED':
            return {
                'status': 'blocked',
                'response': model_response,
                'metadata': {'stage': 'input_output_filters'}
            }

        # Step 2: Check grounding and relevance
        grounding_result = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            source='OUTPUT',
            content=[
                {'text': {'text': reference_doc, 'qualifiers': ['grounding_source']}},
                {'text': {'text': query, 'qualifiers': ['query']}},
                {'text': {'text': model_response, 'qualifiers': ['guard_content']}}
            ]
        )

        grounding_action = grounding_result['action']
        scores = {}
        for assessment in grounding_result.get('assessments', []):
            if 'contextualGroundingPolicy' in assessment:
                for f in assessment['contextualGroundingPolicy']['filters']:
                    scores[f['type']] = {
                        'score': f['score'],
                        'threshold': f['threshold'],
                        'action': f['action']
                    }

        if grounding_action == 'GUARDRAIL_INTERVENED':
            # Determine which check failed
            g_failed = scores.get('GROUNDING', {}).get('action') == 'BLOCKED'
            r_failed = scores.get('RELEVANCE', {}).get('action') == 'BLOCKED'
            
            if g_failed and r_failed:
                status = 'ungrounded_and_irrelevant'
            elif g_failed:
                status = 'ungrounded'
            else:
                status = 'irrelevant'

            return {
                'status': status,
                'response': (
                    "I want to make sure I give you accurate information. "
                    "Let me look into that more carefully. Could you rephrase "
                    "your question or ask about a specific part of your policy?"
                ),
                'metadata': {'stage': 'grounding_check', 'scores': scores}
            }

        return {
            'status': 'success',
            'response': model_response,
            'metadata': {'stage': 'grounding_check', 'scores': scores}
        }

    except Exception as e:
        return {
            'status': 'error',
            'response': 'We encountered a technical issue. Please try again.',
            'metadata': {'error': str(e)}
        }


# Quick test
result = invoke_with_grounding(
    "What is my collision deductible?",
    POLICY_DOCUMENT
)
print(f"Status: {result['status']}")
print(f"Response: {result['response'][:200]}")
print(f"Scores: {result['metadata'].get('scores', {})}")

Status: success
Response: According to your policy document, your collision deductible is **$500**.
Scores: {'GROUNDING': {'score': 0.98, 'threshold': 0.7, 'action': 'NONE'}, 'RELEVANCE': {'score': 1.0, 'threshold': 0.7, 'action': 'NONE'}}


In [24]:
# ============================================================
# Cell 13: End-to-End Grounding Test Suite
# ============================================================

test_cases = [
    ("Grounded answer",
     "What is my liability coverage limit?",
     POLICY_DOCUMENT),
    
    ("Excluded scenario",
     "Am I covered if I use my car for delivery services?",
     POLICY_DOCUMENT),
    
    ("Denied topic (should block at step 1)",
     "Should I invest my settlement in stocks?",
     POLICY_DOCUMENT),
    
    ("PII block (should block at step 1)",
     "My SSN is 123-45-6789, look up my policy",
     POLICY_DOCUMENT),
]

print("=" * 70)
print("GROUNDING TEST SUITE")
print("=" * 70)

for label, query, doc in test_cases:
    result = invoke_with_grounding(query, doc)
    scores = result['metadata'].get('scores', {})
    g_score = scores.get('GROUNDING', {}).get('score', '—')
    r_score = scores.get('RELEVANCE', {}).get('score', '—')
    stage = result['metadata'].get('stage', '—')
    
    icon = {'success': '✅', 'blocked': '🛑', 'ungrounded': '⚠️',
            'irrelevant': '↩️', 'error': '❌'}.get(result['status'], '❓')
    
    print(f"\n{icon} {label}")
    print(f"   Query:     {query[:55]}")
    print(f"   Status:    {result['status']} | Stage: {stage}")
    print(f"   Grounding: {g_score} | Relevance: {r_score}")
    print(f"   Response:  {result['response'][:75]}...")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

GROUNDING TEST SUITE

✅ Grounded answer
   Query:     What is my liability coverage limit?
   Status:    success | Stage: grounding_check
   Grounding: 1.0 | Relevance: 1.0
   Response:  Based on your policy document, your liability coverage limits are:

1. **Bo...

✅ Excluded scenario
   Query:     Am I covered if I use my car for delivery services?
   Status:    success | Stage: grounding_check
   Grounding: 0.91 | Relevance: 1.0
   Response:  No, you are not covered if you use your car for delivery services. 

Accord...

🛑 Denied topic (should block at step 1)
   Query:     Should I invest my settlement in stocks?
   Status:    blocked | Stage: input_output_filters
   Grounding: — | Relevance: —
   Response:  I'm sorry, I can't process that request. Please rephrase your question abou...

🛑 PII block (should block at step 1)
   Query:     My SSN is 123-45-6789, look up my policy
   Status:    blocked | Stage: input_output_filters
   Grounding: — | Relevance: —
   Response:  I'm sorry

## Day 6 Summary

**What we built:** Contextual grounding checks that verify model responses are factually 
supported by source documents and relevant to the customer's question.

**Key findings:**
- Grounding and relevance are **independent checks** — a response can be grounded but 
  irrelevant, or relevant but hallucinated
- The grounding model understands **semantic equivalence** — paraphrasing passes (1.0), 
  fabrication fails (0.04–0.11), even when mixed with grounded facts
- One hallucinated detail **tanks the whole score** — there's no partial credit
- Contextual grounding requires the **ApplyGuardrail API** (two-step flow), not inline 
  tags with invoke_model
- First-pass guardrails (content filters, denied topics, PII, word filters) run at 
  **step 1**, grounding runs at **step 2** — no wasted grounding checks on blocked requests

**Production pattern:** invoke_model → apply_guardrail (two-step)

**What's next RAG Integration — connecting the Knowledge Base from Phase 4 
so reference documents are retrieved automatically instead of passed in manually.